# 1. Package Imports Section

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
import re
import logging

# 2. Dataset Configs

In [0]:
# logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_Suburb_Arrears")

#silver tables config
ds_config = {
        "silver_table": "cpt_utility_catalog.silver.silver_suburb_electricity_billing_cleaned",
        "bronze_table": "cpt_utility_catalog.bronze.bronze_suburb_electricity_billing_raw",
        "changes": {
            "headers": {
                "column_mapping":{
                   "Subcouncil": "subcouncil",
                   "Date": "date",
                   "Suburb":"suburb",
                   "Billing_Class": "billing_class",
                    "Amount": "amount",
                    "Quantity": "quantity",
                    "Number_of_Records": "number_of_records",
                    "Number_of_Contract_accounts": "number_of_contract_accounts",
                }
               
            },
            "columns": {
                 "data_types":{
                    "amount": "decimal(13,2)",
                    "date": "date",
                    "suburb": "string",
                    "number_of_records": "int",
                    "number_of_contract_accounts": "int",
                    "quantity": "int",
                    "billing_class": "string"
                },
                "columns_to_drop": ["ObjectId"],
                "fill_na_value": None, 
            },
            "trim": True,
            "drop_columns": True,
            "drop_duplicates": True,
            "write_to_table": True,
            "rename_headers": True,
            "add_id": True,
            "cast_type": True,
            "col_cleanse": True,
        },
    }

df_new = spark.table(ds_config["bronze_table"])
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]
changes = ds_config["changes"]

logger.info("\t- Silver layer Dam levels table configuration loaded")

# 3. Dataset Cleaning

## 3.1 Dropping Columns